# Persistence Index

Measures how long events last once they start: longest run of consecutive affected months, typical run length and number of calendar months affected per canton and province, combined into a 0-1 index.

Outputs: `persistence_index_*` tables in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Efficient reader for GeoPackage layers
import pyogrio

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step6_persistence_index", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("persistencia iniciada — configuración cargada desde %s", CONFIG_PATH)

# First year of the analysis window
YEAR_MIN = config["date_range"]["start_year"]
# Last year of the analysis window
YEAR_MAX = config["date_range"]["end_year"]
# List of all years in the analysis window
ALL_YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
# The twelve calendar months
ALL_MONTHS = list(range(1, 13))
# Complete timeline of (year, month) pairs in the window
TIMELINE = [(y, m) for y in ALL_YEARS for m in ALL_MONTHS]
# Check the timeline has 132 months (11 years x 12)
assert len(TIMELINE) == 132

# Persistence index settings from the config
PER_CFG = config["persistence_index"]
# Weights of the three persistence components
COMPONENT_WEIGHTS = PER_CFG["component_weights"]
# Number of calendar months used to scale season length
SEASON_DENOM = PER_CFG["season_length_denominator"]
# Check the component weights add up to 1
assert abs(sum(COMPONENT_WEIGHTS.values()) - 1.0) < 1e-3, "los pesos de componentes deben sumar 1.0"
# Log the component weights
logger.info("Pesos de componentes de Persistencia: %s", COMPONENT_WEIGHTS)

## 1. Load data

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the join did not change the number of events
assert len(he) == n_antes
# Check every event received an official name
assert he[["canton_name_official", "province_name_official"]].isna().sum().sum() == 0
# Log that official names were attached by canton code
logger.info("Nombres oficiales incorporados por atributo (canton_code); cobertura 100%%")

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

## 2. Runs of consecutive affected months

In [ ]:
# Lengths of runs of consecutive affected months
def calcular_rachas(afectado: pd.Series) -> list[int]:
    """Lengths of runs of consecutive affected months.
    
    Args:
        afectado: Boolean series, True for months with events.
    
    Returns:
        List with the length of each affected run.
    """
    # Id that changes each time the affected state flips
    cambio = (afectado != afectado.shift(fill_value=False)).cumsum()
    # Sum and size of each run
    rachas = afectado.groupby(cambio).agg(["sum", "size"])
    # Keep only the runs where the unit was affected
    rachas_afectadas = rachas[afectado.groupby(cambio).first()]
    # Return the length of each affected run
    return rachas_afectadas["size"].tolist()


# Compute the three persistence components for one unit
def calcular_persistencia_por_unidad(eventos_mes: set[tuple[int, int]]) -> tuple[int, float, int]:
    """Compute the three persistence components for one administrative unit.
    
    Args:
        eventos_mes: Set of (year, month) pairs with at least one event.
    
    Returns:
        Longest run, mean run length and season length (distinct calendar months affected).
    """
    # Boolean series: was the unit affected in each month of the timeline
    afectado = pd.Series([par in eventos_mes for par in TIMELINE])
    # Lengths of all affected runs
    rachas = calcular_rachas(afectado)
    # Longest run, zero if there are none
    racha_maxima = max(rachas) if rachas else 0
    # Mean run length, zero if there are none
    duracion_media = (sum(rachas) / len(rachas)) if rachas else 0.0
    # Calendar months (1-12) in which the unit was ever affected
    meses_calendario = {m for (y, m) in eventos_mes}
    # Season length: number of distinct calendar months affected
    longitud_temporada = len(meses_calendario)
    # Return the three components
    return racha_maxima, duracion_media, longitud_temporada

## 3. Canton and province level, per hazard and combined

In [ ]:
# Persistence components for every unit of one administrative level
def ejecutar_nivel_persistencia(df: pd.DataFrame, admin_cols: list[str]) -> pd.DataFrame:
    """Compute persistence components for every unit of one administrative level, per hazard and combined.
    
    Args:
        df: Event table with official names.
        admin_cols: Columns that define the administrative level.
    
    Returns:
        Table with one row per unit and hazard view, including units without events (components at zero).
    """
    # Universe of all units that have any historical event
    universo = df[admin_cols].drop_duplicates()

    # Compute the components for one hazard view
    def _para_hazard(sub_df: pd.DataFrame, hazard_label: str) -> pd.DataFrame:
        # List that collects one row per unit
        filas = []
        # Loop over every administrative unit
        for _, unidad in universo.iterrows():
            # Select the events that belong to the unit
            mask = (sub_df[admin_cols] == unidad).all(axis=1)
            # Distinct (year, month) pairs with events for the unit
            eventos_mes = set(map(tuple, sub_df.loc[mask, ["year", "month"]].drop_duplicates().values))
            # Compute the three components for the unit
            racha_max, dur_media, temp_len = calcular_persistencia_por_unidad(eventos_mes)
            # Store the row with the components and the event count
            filas.append({**unidad.to_dict(), "hazard_type": hazard_label,
                          "racha_maxima": racha_max, "duracion_media": round(dur_media, 3),
                          "longitud_temporada": temp_len, "n_eventos": mask.sum()})
        # Return the table for this view
        return pd.DataFrame(filas)

    # One table per hazard type
    partes = [_para_hazard(df[df.hazard_type == h], h) for h in sorted(df.hazard_type.unique())]
    # Add the combined view of all hazards
    partes.append(_para_hazard(df, "TODOS"))
    # Stack all views
    return pd.concat(partes, ignore_index=True)

In [ ]:
# Log the start of the canton-level calculation
logger.info("Calculando componentes de Persistencia a nivel cantón (puede tardar unos segundos)...")
# Compute persistence at canton level
per_canton = ejecutar_nivel_persistencia(he, CANTON_COLS)
# Log the start of the province-level calculation
logger.info("Calculando componentes de Persistencia a nivel provincia...")
# Compute persistence at province level
per_provincia = ejecutar_nivel_persistencia(he, PROVINCE_COLS)
# Log the number of rows produced
logger.info("Cantón: %d filas | Provincia: %d filas", len(per_canton), len(per_provincia))

## 4. Normalization and composite index

In [ ]:
# Normalize the components and combine them into the index
def normalizar_y_combinar(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize the three components and combine them into the persistence index.
    
    Args:
        df: Table with longest run, mean duration and season length.
    
    Returns:
        Copy of the table with the normalized columns and 'persistence_index' in [0, 1].
    """
    # Work on a copy
    df = df.copy()

    # Min-max scaling of one series
    def _minmax(s: pd.Series) -> pd.Series:
        # Range of the series
        rango = s.max() - s.min()
        # A constant series has no range
        if rango == 0:
            # Return zeros for a constant series
            return pd.Series(0.0, index=s.index)
        # Scale to the 0-1 interval
        return (s - s.min()) / rango

    # Normalize the longest run within each hazard view
    df["norm_racha_maxima"] = df.groupby("hazard_type")["racha_maxima"].transform(_minmax)
    # Normalize the mean duration within each hazard view
    df["norm_duracion_media"] = df.groupby("hazard_type")["duracion_media"].transform(_minmax)
    # Scale season length by the calendar-month denominator, capped at 1
    df["norm_longitud_temporada"] = (df["longitud_temporada"] / SEASON_DENOM).clip(upper=1.0)

    # Weighted sum of the three normalized components
    df["persistence_index"] = (
        COMPONENT_WEIGHTS["racha_maxima"] * df["norm_racha_maxima"]
        + COMPONENT_WEIGHTS["duracion_media"] * df["norm_duracion_media"]
        + COMPONENT_WEIGHTS["longitud_temporada"] * df["norm_longitud_temporada"]
    )
    # Check the index is between 0 and 1
    assert df["persistence_index"].between(0, 1).all(), "persistence_index fuera de [0,1]"
    # Return the table with the index
    return df

In [ ]:
# Persistence index at canton level
per_canton = normalizar_y_combinar(per_canton)
# Persistence index at province level
per_provincia = normalizar_y_combinar(per_provincia)
# Log that the index is valid at both levels
logger.info("Índice de Persistencia calculado y verificado en [0,1] para ambos niveles")

## 5. Validation

In [ ]:
# Number of hazard types in the data
n_hazard_types = he.hazard_type.nunique()
# Check every canton appears in every view, including the combined one
assert len(per_canton) == 221 * (n_hazard_types + 1), "cobertura incompleta a nivel cantón"
# Check every province appears in every view
assert len(per_provincia) == 24 * (n_hazard_types + 1), "cobertura incompleta a nivel provincia"

# Check no canton run exceeds the 132-month timeline
assert (per_canton["racha_maxima"] <= 132).all()
# Check no province run exceeds the timeline
assert (per_provincia["racha_maxima"] <= 132).all()

# Check season length never exceeds 12 months
assert (per_canton["longitud_temporada"] <= 12).all()
# Check season length never exceeds 12 months at province level
assert (per_provincia["longitud_temporada"] <= 12).all()

# Check the mean duration never exceeds the longest run
assert (per_canton["duracion_media"] <= per_canton["racha_maxima"] + 1e-9).all()

# Ten cantons with the highest persistence
top10 = per_canton[per_canton.hazard_type == "TODOS"].sort_values("persistence_index", ascending=False).head(10)
# Print the top ten
print(top10[["province_name_official", "canton_name_official", "racha_maxima",
             "duracion_media", "longitud_temporada", "persistence_index"]].to_string(index=False))

## 6. Export

In [ ]:
# Tables to export
tablas = {
    "persistence_index_canton": per_canton,
    "persistence_index_provincia": per_provincia,
}

# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")